
```
docker run -it --rm \
    --name aic_eval \
    --network host \
    --gpus all \
    -e DISPLAY=$DISPLAY \
    -v /tmp/.X11-unix:/tmp/.X11-unix \
    ghcr.io/intrinsic-dev/aic/aic_eval:latest \
    ground_truth:=false start_aic_engine:=true
```
   
   
---   
   Here's the breakdown:

`docker run -it --rm`
   - `docker run` - create and start a new container
   - `-i` (interactive) - keep STDIN open so you can send input
   - `-t` (tty) - allocate a pseudo-terminal (gives you a proper terminal with
     colors, Ctrl+C, etc.)
   - `--rm` - automatically delete the container when it stops (clean it after
     yourself)


`--name aic_eval`     
   Names the container `aic_eval` so you can reference it by name (e.g, 
   `docker exec aic_eval ..., docker stop aic_eval`) instead of a random hex ID.


`--network host`   
   Instead of Docker's default isolated virtual network, the container shares 
   the host's (WSL2's) network stack directly. This means:
      * Container sees the same IP, ports, interfaces as WSL2
      * ROS 2 Zenoh middleware can communicate freely between the container and
        your `pixi` policy running outside
      * This is why it fixed the DNS/networking issue -- no NAT/bridge layer
        to break things.


`--gpus all`          
   Passes all NVIDIA GPUs to the container via the NVIDIA Container Toolkit. 
   Without this, Gazebo would run CPU-only rendering (slow, ugly). With it, the
   RTX 5090 handles physics and rendering.


`-e DISPLAY=$DISPLAY`
   Sets the `DISPLAY` environment variable inside the container to match your
   WSL2 host's value. This tells GUI apps (Gazebo, RViz) where to draw their 
   windows. In WSL2 with WSLg, this is typically `:0`.


`-v /tmp/.X11-unix:/tmp/.X11-unix`
   Bind-mounts the X11 socket directory from WSL2 into the container. The
   `DISPLAY` variable tells apps which display, and this mount gives them access
   to the actual Unix socket file that connects to the WSLg display server.
   Together, these two flags enable GUI windows from inside the container to
   appear on your Windows desktop.   


`ghcr.io/intrinsic-dev/aic/aic_eval:latest`
   The docker image to run. `ghcr.io` = Github Container Registry, 
   `intrinsic-dev/aic` = the AIC org/repo, `aic_eval` = the evaluation image,
   `latest` = most recent tag.


`ground_truth:=false start_aic_engine:=true`
   These are not Docker flags - they're passed as arguments to the container's
   ENTRYPOINT (`/entrypoint.sh`). The entrypoint script forwards them to the
   ROS 2 launch file:

   * `ground_truth:=false` -- don't publish ground-truth TF frames (competition
     mode -- your policy must use camera)
   * `start_aic_engine:=true` -- start the trail engine that orchestrate 3 
     trails, spawns the board/cable, and scores your policy

---
   So in one sentence: it runs the pre-built AIC evaluation environment as a 
   GPU-acelerated container that shares the host network (for ROS 2 comms) and
   display (for Gazebo GUI), then starts the trail engine in non-cheating mode.     

❯ answer in like 1-2 paragraphs, when you say `Docker image`, the `image` here means more like an iso file for like an OS right?

---

   ... A `DOCKER IMAGE` is like an ISO -- it's a read-only snapshot of an entire
   filesystem: OS, libraries, compiled programs, config files, everything needed
   to run the software. The `aic_eval` image is essentially 
   "Ubuntu 24.04 + ROS 2 Kilted + Gazebo + all the AIC evaluation code, 
   pre-built and ready to go" packaged into one downloadable blob.

   When you `docker run` an image, it creates a CONTAINER -- a running instance
   of that image, similar to how booting an ISO gives you a live OS. But unlike
   a VM, containers share the host's kernel (WSL2's Linux kernel in your case),
   so they're much lighter and start in seconds. You can run multiple containers
   from the same image, each isolated, and when you `--rm` them they vanish -
   the original image stays untouched on disk for next time. 




   SCORING RESULTS ON THE RIGHT -- Yes, that's the AIC engine's scoring output
   after running trails. It scores 0 across all tiers because 
   `Model Validation Failed` -- the engine couldn't find any `aic_model` node.
   Nobody was home to control the robot.


   WHAT THE DOCKER CONTAINER DOES -- The eval container does NOT run any 
   policies itself. It only runs the "arena":
      - Gazebo (physics + rendering)
      - Robot controller
      - Sensor fusion (cameras, F/T sensor)
      - Trail engine (spawns board, manages trails, times you)
      - Scoring system

   It then WAITS for your policy to connect as a separate ROS 2 node. When you
   never launched `WaveArm` in Terminal 2, the engine waited 30 seconds, timed
   out, and scored you 0 ("Model validation failed").

THE TWO-PIECE ARCHITECTURE:
   Terminal 1 (Docker):     The arena -- sim, robot, scoring        <-- already running
   Terminal 2 (Pixi):       YOUR policy -- the brain                <-- you never started this

   ...

---

### Terminal 1

In [ ]:
  mkdir -p ~/aic_results

  docker run -it --rm \
    --name aic_eval \
    --network host \
    --gpus all \
    -e DISPLAY=:0 \
    -e WAYLAND_DISPLAY=wayland-0 \
    -e XDG_RUNTIME_DIR=/mnt/wslg/runtime-dir \
    -e PULSE_SERVER=/mnt/wslg/PulseServer \
    -e GALLIUM_DRIVER=d3d12 \
    -e LD_LIBRARY_PATH=/usr/lib/wsl/lib \
    -e AIC_RESULTS_DIR=/aic_results \
    -v /tmp/.X11-unix:/tmp/.X11-unix \
    -v /mnt/wslg:/mnt/wslg \
    -v /usr/lib/wsl:/usr/lib/wsl \
    -v ~/aic_results:/aic_results \
    --device /dev/dxg \
    ghcr.io/intrinsic-dev/aic/aic_eval:latest \
    ground_truth:=false start_aic_engine:=true

### Terminal 2

In [ ]:
  cd ~/projects/Project-Automaton/References/aic
  pixi run ros2 run aic_model aic_model --ros-args -p use_sim_time:=true -p policy:=aic_example_policies.ros.WaveArm


---

 To answer your question: what does success look like?

  Yes, exactly what you described. When WaveArm runs successfully:

  1. In Gazebo: A task board + cable appear in the scene. The robot arm physically moves — waving back and forth (WaveArm doesn't try to
  insert, it just waves)
  2. In Terminal 1 (eval): You see trial progression:
  Trial 1/3 started...
  Trial 1/3 completed.
  Trial 2/3 started...
  ...
  3. At the end: A scoring summary prints, something like:
  Complete Scoring Results
  total: 3
  trial_1:
    tier_1:
      score: 1        ← "model loaded and responded" = success
    tier_2:
      score: ~0-6     ← trajectory quality (low for WaveArm)
    tier_3:
      score: 0        ← insertion (0 because WaveArm doesn't try)
  4. Results file: scoring.yaml saved to ~/aic_results/

  WaveArm is supposed to score low — it just proves the pipeline works end-to-end. CheatCode (with ground_truth:=true) is the one that
  actually inserts and scores high.

---

  WaveArm is done — scored 73.6 pts across 3 trials. Results are in ~/projects/Project-Automaton/aic_results/.

  Next: Run CheatCode (the gold-standard policy that actually inserts the cable). Two terminals in WSL2:

  Terminal 1:
  docker run -it --rm \
    --name aic_eval \
    --network host \
    --gpus all \
    -e DISPLAY=:0 \
    -e WAYLAND_DISPLAY=wayland-0 \
    -e XDG_RUNTIME_DIR=/mnt/wslg/runtime-dir \
    -e PULSE_SERVER=/mnt/wslg/PulseServer \
    -e GALLIUM_DRIVER=d3d12 \
    -e LD_LIBRARY_PATH=/usr/lib/wsl/lib \
    -e AIC_RESULTS_DIR=/aic_results \
    -v /tmp/.X11-unix:/tmp/.X11-unix \
    -v /mnt/wslg:/mnt/wslg \
    -v /usr/lib/wsl:/usr/lib/wsl \
    -v ~/projects/Project-Automaton/aic_results:/aic_results \
    --device /dev/dxg \
    ghcr.io/intrinsic-dev/aic/aic_eval:latest \
    ground_truth:=true start_aic_engine:=true

  Note: ground_truth:=true this time — CheatCode needs it.

  Terminal 2 (once you see Retrying...):
  cd ~/projects/Project-Automaton/References/aic
  pixi run ros2 run aic_model aic_model --ros-args -p use_sim_time:=true -p policy:=aic_example_policies.ros.CheatCode